<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/9-structured-output-multi-step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured Output and Multi-Step Agents

This notebook demonstrates two powerful Vectara agent features:

1. **Structured Output**: Configure agents to return validated JSON conforming to a schema, using the model's native structured outputs capability
2. **Multi-Step Agents**: Define agents with multiple steps and conditional routing, enabling workflow-like behavior

You'll learn how to:
1. Create agents with structured output parsers for guaranteed JSON responses
2. Define multi-step agents with conditional transitions between steps
3. Use step-specific tool restrictions with `allowed_tools`
4. Parse structured output events and step transition events
5. Combine both features to build a classifier-router pattern

## About Vectara

[Vectara](https://vectara.com/) is the Agent Platform for trusted enterprise AI: a unified Agentic RAG platform with built-in multi-modal retrieval, orchestration, and always-on governance. Deploy it on-prem (air-gapped), in your VPC, or as SaaS. Vectara agents deliver grounded answers and safe actions with source citations, step-level audit trails, fine-grained access controls, and real-time policy and factual-consistency enforcement, so teams ship faster with lower risk, and with trusted, production-grade AI agents at scale.

Vectara provides a complete API-first platform for building production RAG and agentic applications:

- **Simple Integration**: RESTful APIs and SDKs (Python, JavaScript) for quick integration into any stack
- **Flexible Deployment**: Choose SaaS, VPC, or on-premises deployment based on your requirements
- **Multi-Modal Support**: Index and search across text, tables, and images from PDFs, documents, and structured data
- **Advanced Retrieval**: Hybrid search combining semantic and keyword matching with state-of-the-art reranking
- **Grounded Generation**: LLM responses with citations and factual consistency scores to reduce hallucinations
- **Enterprise-Ready**: Built-in access controls, audit logging, and compliance (SOC2, HIPAA) from day one

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs)
- Notebook 2: Ingested AI research papers and Vectara documentation

## Setup

In [1]:
import os
import requests
import json
from datetime import datetime

api_key = os.environ['VECTARA_API_KEY']

research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

print("Setup complete.")

Setup complete.


In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)


In [3]:
def chat_with_agent(agent_key, session_key, message, show_events=False):
    """Send a message to an agent and return its terminal output.

    For a single-step agent with a structured output parser, this returns
    the validated dict from `structured_output`. For a multi-step agent
    that transitions classifier -> handler, we want the handler's final
    `agent_output`, not the classifier's intermediate `structured_output`.
    We walk every event and keep the last terminal output we saw, which
    naturally picks the right one in both cases.
    """
    message_data = {
        "messages": [{"type": "text", "content": message}],
        "stream_response": False
    }
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=message_data)

    if response.status_code != 201:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    event_data = response.json()
    if show_events:
        print("\n--- Agent Events ---")
        for event in event_data.get('events', []):
            event_type = event.get('type', 'unknown')
            if event_type == 'step_transition':
                print(f"  Step transition: {event.get('from_step', '?')} -> {event.get('to_step', '?')}")
            elif event_type == 'structured_output':
                print(f"  Structured output ({event.get('schema_name', '?')}): "
                      f"{json.dumps(event.get('content', {}))[:200]}")
            elif event_type == 'tool_input':
                print(f"  Tool call: {event.get('tool_configuration_name', '?')}")
            elif event_type == 'tool_output':
                print(f"  Tool response: {event.get('tool_configuration_name', '?')}")
            elif event_type == 'agent_output':
                print(f"  Agent output: {event.get('content', '')[:200]}...")
        print("---\n")

    last_output = None
    for event in event_data.get('events', []):
        etype = event.get('type')
        if etype == 'structured_output':
            last_output = event.get('content')
        elif etype == 'agent_output':
            last_output = event.get('content', 'No content')
    return last_output if last_output is not None else "No output found"


## Part 1: Structured Output

### Step 1: Create an Agent with Structured Output

Instead of free-form text, you can configure an agent to return validated JSON conforming to a schema.
The `StructuredOutputParser` uses the model's native structured outputs capability to guarantee
valid JSON on every response. This is useful for entity extraction, classification, and downstream
API integrations where you need a predictable response format.

When configured, the agent emits `structured_output` events (instead of `agent_output`) containing
the validated JSON and the schema name.

In [4]:
extraction_agent_config = {
    "name": "Research Entity Extractor",
    "description": "Extracts structured entities from research questions",
    "model": {"name": "gpt-4o"},
    "first_step_name": "extract",
    "steps": {
        "extract": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "extraction_instructions",
                    "template": (
                        "You are a research entity extraction agent. Given a user question about AI research, "
                        "search the corpus for relevant information, then extract structured entities from both "
                        "the question and retrieved content.\n\n"
                        "Identify:\n"
                        "- The main research topics mentioned\n"
                        "- Specific techniques or methods referenced\n"
                        "- A brief answer to the question based on retrieved content\n"
                        "- A confidence level for your answer"
                    )
                }
            ],
            "output_parser": {
                "type": "structured",
                "json_schema": {
                    "name": "research_extraction",
                    "description": "Structured extraction of research entities and answer",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "topics": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "techniques": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "answer": {"type": "string"},
                            "confidence": {
                                "type": "string",
                                "enum": ["high", "medium", "low"]
                            }
                        },
                        "required": ["topics", "techniques", "answer", "confidence"],
                        "additionalProperties": False
                    }
                }
            }
        }
    },
    "tool_configurations": {
        "research_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": research_corpus_key}],
                    "limit": 10
                }
            }
        }
    }
}

extraction_agent_key = delete_and_create_agent(extraction_agent_config, "Research Entity Extractor")

Created agent 'Research Entity Extractor' (key: agt_research_entity_extractor_762d)


### Step 2: Test Structured Output

Let's send queries and verify the agent returns validated JSON matching our schema every time.

In [5]:
session_config = {"name": f"Extraction Demo {datetime.now().strftime('%Y%m%d-%H%M%S')}"}
response = requests.post(
    f"{BASE_URL}/agents/{extraction_agent_key}/sessions",
    headers=headers,
    json=session_config
)

extraction_session_key = None
if response.status_code == 201:
    extraction_session_key = response.json()["key"]
    print(f"Session created: {extraction_session_key}")
else:
    print(f"Error creating session: {response.text}")

Session created: ase_extraction_demo_20260420-123500_ac88


In [6]:
result = chat_with_agent(
    extraction_agent_key,
    extraction_session_key,
    "What techniques does RAG use to reduce hallucinations in language models?",
    show_events=True
)

print("Structured Result:")
print(json.dumps(result, indent=2))

# Access fields programmatically
if isinstance(result, dict):
    print(f"\nTopics: {result.get('topics', [])}")
    print(f"Techniques: {result.get('techniques', [])}")
    print(f"Confidence: {result.get('confidence', 'N/A')}")


--- Agent Events ---
  Tool call: research_search
  Tool response: research_search
  Structured output (research_extraction): {"topics": ["hallucinations in large language models", "RAG (Retrieval-Augmented Generation)"], "techniques": ["retrieval-augmented generation", "self-reflection", "fine-grained detection and editing"
---

Structured Result:
{
  "topics": [
    "hallucinations in large language models",
    "RAG (Retrieval-Augmented Generation)"
  ],
  "techniques": [
    "retrieval-augmented generation",
    "self-reflection",
    "fine-grained detection and editing",
    "hallucination detection models",
    "NLI-based models",
    "G-eval using GPT-4",
    "RAGTruth corpus"
  ],
  "answer": "RAG uses techniques such as retrieval-augmented generation to reduce hallucinations, improving factual consistency by cross-referencing retrieved documents. The approach leverages the explicit retrieval of knowledge to ground responses in factual evidence, thus reducing the propensity fo

In [7]:
# Second query — same schema, different content
result2 = chat_with_agent(
    extraction_agent_key,
    extraction_session_key,
    "How do dense retrieval models compare to sparse retrieval methods like BM25?",
    show_events=True
)

print("Structured Result:")
print(json.dumps(result2, indent=2))


--- Agent Events ---
  Tool call: research_search
  Tool response: research_search
  Structured output (research_extraction): {"topics": ["dense retrieval models", "sparse retrieval methods", "information retrieval", "entity retrieval"], "techniques": ["BM25 (sparse retrieval method)", "dense passage retrieval", "word overla
---

Structured Result:
{
  "topics": [
    "dense retrieval models",
    "sparse retrieval methods",
    "information retrieval",
    "entity retrieval"
  ],
  "techniques": [
    "BM25 (sparse retrieval method)",
    "dense passage retrieval",
    "word overlap-based retrieval",
    "dense representations"
  ],
  "answer": "Dense retrieval models, like those employing dense passage retrieval techniques, generally offer superior performance in open-domain question answering by effectively capturing semantic information. They outperform sparse retrieval methods like BM25, which are based on word overlap and may excel in heavily entity-centric datasets like FEVER, 

## Part 2: Multi-Step Agent with Classifier-Router Pattern

### Step 3: Create a Multi-Step Agent

A common production pattern is the **classifier-router**: a first step classifies the user's
intent using structured output, then conditional routing sends the conversation to a specialized
handler step. Each step can have its own instructions and restricted tool access via `allowed_tools`.

Key concepts:
- **`steps` map**: Define named steps (replaces the deprecated `first_step` field)
- **`first_step_name`**: References the entry point step in the map
- **`next_steps`**: Conditional transitions using UserFn expressions (`get('$.output.field') == 'value'`)
- **`allowed_tools`**: Restrict which tools each step can use (empty list = no tools)

In [8]:
classifier_agent_config = {
    "name": "Research Assistant Router",
    "description": "Multi-step agent that classifies queries and routes to specialized handlers",
    "model": {"name": "gpt-4o"},
    "first_step_name": "classifier",
    "steps": {
        "classifier": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "classifier_instructions",
                    "template": (
                        "You are a query classifier. Analyze the user's question and classify it into one of "
                        "these intents:\n\n"
                        '- "research": Questions about academic research, theoretical concepts, algorithms, or '
                        "papers (e.g., 'How does attention work in transformers?')\n"
                        '- "implementation": Questions about using Vectara APIs, SDKs, configuration, or building '
                        "applications (e.g., 'How do I set up hybrid search?')\n"
                        '- "comparison": Questions that compare, contrast, or evaluate multiple approaches '
                        "(e.g., 'BM25 vs dense retrieval?')\n\n"
                        "Classify based on the primary intent. Do not use any tools."
                    )
                }
            ],
            "output_parser": {
                "type": "structured",
                "json_schema": {
                    "name": "intent_classification",
                    "description": "Classifies user query intent for routing",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "intent": {
                                "type": "string",
                                "enum": ["research", "implementation", "comparison"]
                            },
                            "reasoning": {"type": "string"}
                        },
                        "required": ["intent", "reasoning"],
                        "additionalProperties": False
                    }
                }
            },
            "allowed_tools": [],
            "next_steps": [
                {
                    "condition": "get('$.output.intent') == 'research'",
                    "step_name": "research_handler"
                },
                {
                    "condition": "get('$.output.intent') == 'implementation'",
                    "step_name": "implementation_handler"
                },
                {
                    "step_name": "comparison_handler"
                }
            ]
        },
        "research_handler": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "research_instructions",
                    "template": (
                        "You are a research expert. The user's question has been classified as research-oriented. "
                        "Search the research papers corpus for relevant academic content and provide a thorough, "
                        "well-cited answer grounded in the retrieved papers. Focus on theoretical foundations, "
                        "key findings, and methodology."
                    )
                }
            ],
            "output_parser": {"type": "default"},
            "allowed_tools": ["research_search"]
        },
        "implementation_handler": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "implementation_instructions",
                    "template": (
                        "You are an implementation expert. The user's question has been classified as "
                        "implementation-oriented. Search the Vectara documentation corpus for relevant guides "
                        "and provide specific, actionable implementation guidance with API examples and "
                        "configuration details."
                    )
                }
            ],
            "output_parser": {"type": "default"},
            "allowed_tools": ["docs_search"]
        },
        "comparison_handler": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "comparison_instructions",
                    "template": (
                        "You are a comparison analyst. The user's question has been classified as a comparison. "
                        "Search both the research papers and documentation to find relevant information. "
                        "Provide a balanced comparison with evidence from both academic research and "
                        "practical documentation."
                    )
                }
            ],
            "output_parser": {"type": "default"},
            "allowed_tools": ["research_search", "docs_search"]
        }
    },
    "tool_configurations": {
        "research_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": research_corpus_key}],
                    "limit": 10
                }
            }
        },
        "docs_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": docs_corpus_key}],
                    "limit": 10
                }
            }
        }
    }
}

classifier_agent_key = delete_and_create_agent(classifier_agent_config, "Research Assistant Router")

Created agent 'Research Assistant Router' (key: agt_research_assistant_router_4eed)


### Step 4: Test Multi-Step Routing

Let's test with three different query types and observe the step transitions in the event stream.
Each test uses a fresh session to avoid context from prior turns influencing classification.

In [9]:
# Create session for Test 1
response = requests.post(
    f"{BASE_URL}/agents/{classifier_agent_key}/sessions",
    headers=headers,
    json={"name": f"Router Test 1 {datetime.now().strftime('%Y%m%d-%H%M%S')}"}
)
if response.status_code != 201:
    print(f"Error creating session: {response.text}")
session_key_1 = response.json()["key"]

print("=" * 60)
print("TEST 1: Research query (expect: classifier -> research_handler)")
print("=" * 60)

query = "What are the key innovations in the original transformer architecture's attention mechanism?"
print(f"User: {query}\n")

result = chat_with_agent(classifier_agent_key, session_key_1, query, show_events=True)
print(f"Response:\n{result}")

TEST 1: Research query (expect: classifier -> research_handler)
User: What are the key innovations in the original transformer architecture's attention mechanism?


--- Agent Events ---
  Structured output (intent_classification): {"intent": "research", "reasoning": "The question asks about the 'key innovations in the original transformer architecture's attention mechanism', which pertains to understanding the theoretical conce
  Step transition: classifier -> research_handler
  Tool call: research_search
  Tool response: research_search
  Agent output: The key innovations in the attention mechanism of the original Transformer architecture, as introduced by Vaswani et al. in their seminal paper "Attention is All You Need," include the following:

1. ...
---

Response:
The key innovations in the attention mechanism of the original Transformer architecture, as introduced by Vaswani et al. in their seminal paper "Attention is All You Need," include the following:

1. **Self-Attention Mech

In [10]:
# New session for independent test
response = requests.post(
    f"{BASE_URL}/agents/{classifier_agent_key}/sessions",
    headers=headers,
    json={"name": f"Router Test 2 {datetime.now().strftime('%Y%m%d-%H%M%S')}"}
)
if response.status_code != 201:
    print(f"Error creating session: {response.text}")
session_key_2 = response.json()["key"]

print("=" * 60)
print("TEST 2: Implementation query (expect: classifier -> implementation_handler)")
print("=" * 60)

query = "How do I configure hybrid search with reranking in Vectara's API?"
print(f"User: {query}\n")

result = chat_with_agent(classifier_agent_key, session_key_2, query, show_events=True)
print(f"Response:\n{result}")

TEST 2: Implementation query (expect: classifier -> implementation_handler)
User: How do I configure hybrid search with reranking in Vectara's API?


--- Agent Events ---
  Structured output (intent_classification): {"intent": "implementation", "reasoning": "The question specifically asks for guidance on configuring a feature (hybrid search with reranking) using Vectara's API, which falls under the implementation
  Step transition: classifier -> implementation_handler
  Tool call: docs_search
  Tool response: docs_search
  Agent output: To configure hybrid search with reranking using Vectara's API, follow these steps:

1. **Hybrid Search Configuration**:
   - Adjust the `lexical_interpolation` value to determine the balance between n...
---

Response:
To configure hybrid search with reranking using Vectara's API, follow these steps:

1. **Hybrid Search Configuration**:
   - Adjust the `lexical_interpolation` value to determine the balance between neural and lexical search. This value c

In [11]:
# New session
response = requests.post(
    f"{BASE_URL}/agents/{classifier_agent_key}/sessions",
    headers=headers,
    json={"name": f"Router Test 3 {datetime.now().strftime('%Y%m%d-%H%M%S')}"}
)
if response.status_code != 201:
    print(f"Error creating session: {response.text}")
session_key_3 = response.json()["key"]

print("=" * 60)
print("TEST 3: Comparison query (expect: classifier -> comparison_handler)")
print("=" * 60)

query = "How does semantic search compare to keyword-based search for finding relevant documents?"
print(f"User: {query}\n")

result = chat_with_agent(classifier_agent_key, session_key_3, query, show_events=True)
print(f"Response:\n{result}")

TEST 3: Comparison query (expect: classifier -> comparison_handler)
User: How does semantic search compare to keyword-based search for finding relevant documents?


--- Agent Events ---
  Structured output (intent_classification): {"intent": "comparison", "reasoning": "The question is directly asking for a comparison between semantic search and keyword-based search in terms of their effectiveness in finding relevant documents. 
  Step transition: classifier -> comparison_handler
  Tool call: research_search
  Tool call: docs_search
  Tool response: research_search
  Tool response: docs_search
  Agent output: Semantic search and keyword-based search are two distinct approaches to retrieving relevant documents, each with its own advantages and limitations.

### Keyword-Based Search

Keyword-based search rel...
---

Response:
Semantic search and keyword-based search are two distinct approaches to retrieving relevant documents, each with its own advantages and limitations.

### Keyword-Bas

## Cleanup (Optional)

Delete the agents created in this notebook:

In [12]:
for key, name in [(extraction_agent_key, "Research Entity Extractor"),
                  (classifier_agent_key, "Research Assistant Router")]:
    if key:
        response = requests.delete(f"{BASE_URL}/agents/{key}", headers=headers)
        if response.status_code == 204:
            print(f"Deleted agent: {name}")
        else:
            print(f"Error deleting {name}: {response.text}")

Deleted agent: Research Entity Extractor
Deleted agent: Research Assistant Router
